# Phase 2 GSM8K analysis — baseline vs fixed recirculation

Walkthrough order: **1B variant first**, then **4B recirculation vs 4B baseline**, then the diagnostic sweep. Every graph is followed by its interpretation; every number below regenerates from frozen data.

**Inputs (frozen, committed):** `data/pair_1b_paired.csv`, `data/pair_4b_paired.csv`, `data/sweep_4b_100.csv`, `data/runs.json` (derivatives of the immutable run dirs).
**Code:** `../analysis/gsm8k_phase2.py` — all stats/figures live there.
**Reproduce:** `python ../analysis/gsm8k_phase2.py` from `reports/`.

Verdict up front: no significant accuracy effect at either scale (McNemar p 0.74 / 0.26); massive output churn (90% / 69% changed).

In [1]:
import sys
sys.path.insert(0, '..')
from analysis.gsm8k_phase2 import (load_pair, load_sweep, pair_stats,
    summary_table, figure_a, figure_b, figure_c, figure_d,
    figure_e, figure_f, figure_g, load_sweep_transitions)
print(summary_table('data'))

| model | base acc | recirc acc | Δ | rescued | regressed | McNemar p |
|---|---|---|---|---|---|---|
| 1B | 0.0167 | 0.0144 | -0.0023 | 17 | 20 | 0.742 |
| 4B | 0.2942 | 0.2775 | -0.0167 | 163 | 185 | 0.260 |


## 1B variant — Gemma3 1B PT, full test set (n=1319)

Paper configuration (11→4, α=0.15 convex) against the vLLM baseline, greedy two-stage Kojima.

In [2]:
figure_a('data', 'figures/figA_1b.png', only='pair_1b')

### Fig A-1B interpretation

Baseline 0.0167 [0.0110, 0.0251] vs recirculation 0.0144 [0.0092, 0.0224]: the Wilson intervals overlap almost entirely and the delta (−0.0023) is 3 examples out of 1319. Only 2 examples are correct under both conditions — a floor effect dominates: the 1B base model is too weak at GSM8K for any intervention signal to appear in accuracy. Nothing here can confirm or refute the intervention; it only bounds the damage (no collapse).

In [3]:
figure_b('data', 'figures/figB_1b.png', only='pair_1b')

### Fig B-1B interpretation

Flips are symmetric (20 regressed vs 17 rescued, McNemar p=0.74) against 1280 stably-wrong examples: churn without direction. Notably, 90.4% of all outputs changed textually (see §C lengths) while verdicts barely moved — the intervention rewrites reasoning traces far more often than it flips correctness.

In [4]:
figure_c('data', 'figures/figC_1b.png', only='pair_1b')

### Fig C-1B interpretation

Length profiles are similar with recirculation running slightly shorter; both conditions show the short-extraction-answer mode plus a reasoning-length tail. Length is not the mechanism of change — content is. **1B verdict:** null result at the accuracy floor; mechanics (not capability) is all this scale can test.

## 4B — recirculations vs baseline (n=1319)

Paper configuration (18→9, α=0.15, β=1.0 nonconvex) against the vLLM baseline. This is the scale the paper reports gains at.

In [5]:
figure_a('data', 'figures/figA_4b.png', only='pair_4b')

### Fig A-4B interpretation

Baseline 0.2942 [0.2702, 0.3193] vs recirculation 0.2775 [0.2540, 0.3023]: intervals overlap and the delta (−0.0167, −22 net) is not significant. Unlike 1B, the baseline is capable enough for an effect to be visible — and none appears in accuracy. The negative sign must not be over-read either: it is noise (p=0.26).

In [6]:
figure_b('data', 'figures/figB_4b.png', only='pair_4b')

### Fig B-4B interpretation

203 examples are correct under both conditions (a stable capable core), while 185 regressed against 163 rescued (McNemar p=0.26). 69.3% of outputs changed. The symmetry of the off-diagonal mass is the key visual: recirculation moves a large number of verdicts in both directions and they cancel. Any future variant must break this symmetry, not just increase churn.

In [7]:
figure_c('data', 'figures/figC_4b.png', only='pair_4b')

### Fig C-4B interpretation

Near-identical profiles including the bare-number spike at ~0–20 chars in both conditions: stage-2 extraction works the same way with and without recirculation, so the verdict flips in Fig B come from different reasoning content, not from truncation or length artifacts. **4B verdict:** null net effect with decisive trajectory churn — the result that motivates the two-pass schedule experiment.

## Diagnostic sweep — 4B alpha × layers (n=100 first-subset)

18 cells (16-cell grid plus 2 heatmap-completing runs at α=0.10): alphas {0.04, 0.07, 0.10, 0.15} × pairs {(18,9) paper, (16,9), (20,9), (18,7)}, β=1.0, dest-L2, no ramp. Baseline on the same 100: 0.25. Exploratory only (plan §43): the official configuration stays the paper's regardless of this table.

In [8]:
sweep = load_sweep('data')
print(f"{'tag':14s} {'acc':>5s} {'delta':>6s} {'res':>4s} {'reg':>4s} {'p':>6s}")
for r in sorted(sweep, key=lambda r: -r['accuracy']):
    print(f"{r['tag']:14s} {r['accuracy']:5.2f} {r['delta']:+6.2f} {r['rescued']:4d} {r['regressed']:4d} {r['mcnemar_p']:6.3f}")
figure_d('data', 'figures/figD_alpha_sweep.png')

tag              acc  delta  res  reg      p
a010_s18_d7     0.38  +0.13   18    5  0.012
a004_s20_d9     0.36  +0.11   15    4  0.022
a015_s20_d9     0.35  +0.10   17    7  0.066
a007_s18_d9     0.34  +0.09   18    9  0.124
a004_s18_d7     0.33  +0.08   14    6  0.117
a007_s18_d7     0.33  +0.08   13    5  0.099
a007_s20_d9     0.33  +0.08   16    8  0.153
a010_s16_d7     0.33  +0.08   16    8  0.153
a010_s18_d9     0.33  +0.08   16    8  0.153
a010_s20_d7     0.33  +0.08   14    6  0.117
a004_s16_d9     0.32  +0.07   14    7  0.190
a015_s16_d9     0.31  +0.06   18   12  0.361
a004_s18_d9     0.30  +0.05   10    5  0.302
a010_s20_d9     0.30  +0.05   15   10  0.424
a015_s18_d7     0.29  +0.04   17   13  0.584
a015_s18_d9     0.29  +0.04   15   11  0.556
a007_s16_d9     0.25  +0.00   10   10  0.823
a010_s16_d9     0.24  -0.01   10   11  1.000


### Fig D interpretation

![Figure D](figures/figD_alpha_sweep.png)

16 of 18 cells beat the same-100 baseline and the surface is smooth rather than spiky — the intervention *can* produce positive paired deltas here (top cell +0.13, nominal p=0.012). Two disciplines apply: (1) ×16 Bonferroni erases significance; (2) subset difficulty differs from the full test, and subset peeking must not select configurations. s16→d9 is weak twice (0.25, 0.24); the paper pair peaks at α=0.07 on this subset. Takeaway: response surface exists — consistent with the full-scale churn — but nothing here upgrades the null verdict.

In [9]:
figure_e('data', 'figures/figE_sweep_deltas.png')

### Fig E interpretation — every cell vs the baseline

![Figure E](figures/figE_sweep_deltas.png)

This is the head-to-head the sweep exists for: each bar is the *paired* delta of one recirculation configuration against the 0.25 same-100 baseline (identical 100 examples, example_id join). 16 of 18 bars point right (one neutral, one barely left); s16→d9 is the weakest pair at every alpha — the one systematic layer signal in the grid. The two stars mark nominal p<0.05, but with 18 looks the honest reading is Bonferroni-n.s. across the board: suggestive surface, no selectable winner. That is why the official configuration stays the paper's (18,9,0.15) and why Fig E lives in the diagnostic section, not the results section.

### Fig F — paired transitions per sweep cell

Same 18 cells as transition counts against the same-100 baseline (frozen in `data/sweep_4b_100_transitions.csv`). The table is exact; Figure F plots rescued vs regressed so asymmetry is visible at a glance.

In [10]:
tr = load_sweep_transitions('data')
print(f"{'tag':14s} {'cc':>3s} {'cw':>3s} {'wc':>3s} {'ww':>3s}")
for r in tr:
    print(f"{r['tag']:14s} {r['cc']:3d} {r['cw']:3d} {r['wc']:3d} {r['ww']:3d}")
figure_f('data', 'figures/figF_sweep_transitions.png')

tag             cc  cw  wc  ww
a004_s16_d9     18   7  14  61
a004_s18_d7     19   6  14  61
a004_s18_d9     20   5  10  65
a004_s20_d9     21   4  15  60
a007_s16_d9     15  10  10  65
a007_s18_d7     20   5  13  62
a007_s18_d9     16   9  18  57
a007_s20_d9     17   8  16  59
a010_s16_d7     17   8  16  59
a010_s16_d9     14  11  10  65
a010_s18_d7     20   5  18  57
a010_s18_d9     17   8  16  59
a010_s20_d7     19   6  14  61
a010_s20_d9     15  10  15  60
a015_s16_d9     13  12  18  57
a015_s18_d7     12  13  17  58
a015_s18_d9     14  11  15  60
a015_s20_d9     18   7  17  58


![Figure F](figures/figF_sweep_transitions.png)

Every point sits above the null diagonal — on this subset, no configuration regresses more than it rescues — but the cloud hugs the diagonal instead of breaking away: small symmetric-ish churn, matching the full-scale pattern at lower amplitude. The paper cell (blue, 15 rescued / 11 regressed) sits mid-cloud, perfectly ordinary. Two cells touch the diagonal (a007_s16_d9, a010_s16_d9: 10/10 and 10/11) — the same s16→d9 pair that is weakest in accuracy. Read together, Figures D–F say one consistent thing: the intervention moves verdicts around without moving the mean.

In [11]:
figure_g('data', 'figures/figG_layer_heatmap.png')

### Fig G interpretation — the full local heatmap

![Figure G](figures/figG_layer_heatmap.png)

Complete 3×2 window (sources {16,18,20} × destinations {7,9}) at α=0.10 — every cell present, no holes. The s18 row is hottest on both panels (0.38/+0.13 at d7, 0.33/+0.08 at d9); s16→d9 is the lone cold cell (−0.01), consistent with s16→d9 underperforming at every alpha in Figure D. Destination 7 beats destination 9 at every source — a small systematic destination effect worth carrying into any dev-split sweep. Same honesty terms as the rest of this section: n=100, subset difficulty, no selection.

## Second schedule — two-pass (n=100, top-3 sweep cells + paper config)

Same-step source via capture pass + truncated rerun (see the methods schematic, `figures/schematic_recirculation_schedules.png`). Frozen in `data/twopass_4b_100.csv`; compared against the same 100-sample baseline AND the matching cross-step cells.

In [12]:
import csv
with open('data/twopass_4b_100.csv', newline='') as f:
    tp = list(csv.DictReader(f))
print(f"{'tag':16s} {'acc':>5s} {'d_base':>7s} {'d_cross':>7s}")
for r in tp:
    print(f"{r['tag']:16s} {float(r['accuracy']):5.2f} {float(r['delta_vs_base']):+7.2f} {float(r['delta_vs_cross']):+7.2f}")

tag                acc  d_base d_cross
2p_a010_s18_d7    0.31   +0.06   -0.07
2p_a004_s20_d9    0.35   +0.10   -0.01
2p_a015_s20_d9    0.37   +0.12   +0.02
2p_a015_s18_d9    0.33   +0.08   +0.04


### Two-pass interpretation

Two-pass behaves like cross-step within noise (−0.07…+0.04 head-to-head): every cell beats the same-100 baseline (+0.06…+0.12), none dominates its cross-step twin. The schedule variant therefore does not obviously explain the paper gap — reported here precisely so that hypothesis is not funded blindly. Readout-from-rerun was the documented choice; readout-from-normal remains an untested branch.

## Paired statistics (Wilson CI + McNemar, plan §49)

In [13]:
for tag, label in (('pair_1b', '1B'), ('pair_4b', '4B')):
    rows, _ = load_pair('data', tag)
    s = pair_stats(rows)
    lb, ub = s['baseline_ci']
    lt, ut = s['treatment_ci']
    print(f"{label}: base {s['baseline_accuracy']:.4f} [{lb:.4f}, {ub:.4f}] "
          f"vs recirc {s['treatment_accuracy']:.4f} [{lt:.4f}, {ut:.4f}] "
          f"| McNemar chi2={s['mcnemar_chi2']:.3f} p={s['mcnemar_p']:.3f}")
print('\nChanged examples (any output/parse/verdict difference):')
for tag, label in (('pair_1b', '1B'), ('pair_4b', '4B')):
    rows, _ = load_pair('data', tag)
    s = pair_stats(rows)
    print(f"{label}: {s['n_changed']}/{s['n']} = {s['n_changed']/s['n']:.1%}")

1B: base 0.0167 [0.0110, 0.0251] vs recirc 0.0144 [0.0092, 0.0224] | McNemar chi2=0.108 p=0.742
4B: base 0.2942 [0.2702, 0.3193] vs recirc 0.2775 [0.2540, 0.3023] | McNemar chi2=1.267 p=0.260

Changed examples (any output/parse/verdict difference):
1B: 1192/1319 = 90.4%
4B: 914/1319 = 69.3%


## Bottom line

- 1B: null at the floor (mechanics proven, capability untestable).
- 4B: null net with symmetric churn (the finding).
- Sweep: surface responds on a subset (the lead).
- Next: two-pass schedule variant, then dev-split sweeps — not test-set tuning. Changed-question digests in `comparisons/` are the inspection starting point.